In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from fastapi.testclient import TestClient
from src.inference.main import app
client = TestClient(app)
client.__enter__()
client.get("/health").json()

In [ ]:
risky = {"transaction_id": "demo_risky", "user_id": 42, "amount": 1450.0, "hour": 3, "merchant_category": 7, "is_foreign": 1, "card_present": 0, "device_change": 1, "distance_from_home_km": 900.0, "seconds_since_last_txn": 25.0}
normal = {"transaction_id": "demo_normal", "user_id": 42, "amount": 35.0, "hour": 14, "merchant_category": 2, "is_foreign": 0, "card_present": 1, "device_change": 0, "distance_from_home_km": 3.0, "seconds_since_last_txn": 40000.0}
for txn in (risky, normal):
    r = client.post("/score", json=txn).json()
    print(r["transaction_id"], r["risk_score"], r["risk_level"], r["decision"], r["latency_ms"])

In [ ]:
import pandas as pd
pd.DataFrame(client.post("/score", json=risky).json()["top_reasons"])

In [ ]:
import time
import numpy as np
sample = pd.read_parquet("data/raw/transactions.parquet").sample(1000, random_state=1)
payloads = sample.drop(columns=["event_timestamp", "is_fraud"]).to_dict("records")
http_ms, engine_ms = [], []
for p in payloads:
    s = time.perf_counter()
    body = client.post("/score", json=p).json()
    http_ms.append((time.perf_counter() - s) * 1000)
    engine_ms.append(body["latency_ms"]["total"])
summary = pd.DataFrame({"end_to_end_ms": http_ms, "engine_ms": engine_ms}).describe(percentiles=[0.5, 0.95, 0.99]).T.round(3)
summary

In [ ]:
print("Under 50 ms at p99:", bool(np.percentile(http_ms, 99) < 50))

In [ ]:
client.__exit__(None, None, None)